# Dataset

## RealCamVid / Internal Dataset

In [ ]:
import torch
from src.options import opt_dict
# from src.data.realcamvid_dataset import RealcamvidDataset
from src.data.internal_dataset import InternalDataset

opt = opt_dict["wan2.1_t2v_1.3b"]
opt.load_da3_cam, opt.load_depth, opt.load_conf = True, True, True
# dataset = RealcamvidDataset(opt, training=True)
dataset = InternalDataset(opt, training=False)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=dataset.collate_fn)


In [ ]:
data = next(iter(dataloader))

def _multiclip_batch(data):
    prompts = data.get("prompt", None)
    if not isinstance(prompts[0], list):  # one-clip
        return data, None

    # Support dynamic num_clips: only check batch_size=1, not the exact num_clips
    assert len(prompts) == 1 and len(prompts[0]) <= opt.num_clips  # only support batch_size=1 for multi-clip inputs

    new_data = dict(data)
    clip_frame_lens = []
    for key in ["image", "C2W", "fxfycxcy", "depth", "conf"]:
        if key not in data:
            continue
        value = data[key]  # a list (batch) of tuple (clip)
        clip_frame_lens = [[clip.shape[0] for clip in sample] for sample in value]
        new_data[key] = torch.stack([torch.cat(sample, dim=0) for sample in value], dim=0)  # (B=1, sum(F_clip), ...)

    if len(clip_frame_lens) == 0:
        return new_data, None

    clip_frame_lens = torch.tensor(clip_frame_lens, dtype=torch.long)  # (B=1, num_clips)
    clip_latent_lens = []
    for i in range(clip_frame_lens.shape[1]):
        if i == 0:  # first clip keeps the first image latent
            clip_latent_len = 1 + int(round((clip_frame_lens[0, 0:1].item() - 1) / opt.compression_ratio[0]))
            clip_latent_lens.append(clip_latent_len)
        elif i == clip_frame_lens.shape[1] - 1:  # last clip
            all_latent_len = 1 + int(round((clip_frame_lens[0, :].sum().item() - 1) / opt.compression_ratio[0]))
            clip_latent_len = all_latent_len - sum(clip_latent_lens)  # the last clip takes all remaining latents
            clip_latent_lens.append(clip_latent_len)
        else:  # middle clips
            clip_latent_len = int(round(clip_frame_lens[0, i:i+1].sum().item() / opt.compression_ratio[0]))
            clip_latent_lens.append(clip_latent_len)
    clip_latent_lens = torch.tensor(clip_latent_lens, dtype=torch.long)[None, :]  # (B=1, num_clips)

    return new_data, clip_latent_lens

data, clip_latent_lens = _multiclip_batch(data)
for k, v in data.items():
    print(f"{k}: {v.shape if isinstance(v, torch.Tensor) else v}")
print(f"clip_latent_lens: {clip_latent_lens}")


In [ ]:
import imageio.v2 as iio
from src.utils import *

I = 0

print(data["uid"][I])
iio.mimwrite("temp.mp4", tensor_to_video(data["image"][I]), fps=16)
print(data["C2W"][0, 0], data["fxfycxcy"][0, 0])

if "depth" in data:
    depths, C2W, fxfycxcy = data["depth"], data["C2W"], data["fxfycxcy"]
    print("Depth min & max:", depths[I].min(), depths[I].max())
    iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(1./depths[I])), fps=16)

    xyzs = unproject_depth(depths[I:I+1, ...], C2W[I:I+1, ...], fxfycxcy[I:I+1, ...])[0]
    xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
    xyzs = rearrange(xyzs, "c f h w -> f c h w")
    iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs), fps=16)

    # save_xyz_rgb_as_ply("temp.ply", xyzs, data["image"][I], ratio=0.02)
    points, colors = filter_da3_points(
        data["image"][I], data["depth"][I], data["conf"][I],
        data["C2W"][I], data["fxfycxcy"][I],
        conf_thresh_percentile=0.4,
    )
    save_xyz_rgb_as_ply("temp.ply", points, colors, ratio=0.01)

if "conf" in data:
    confs = data["conf"][I]
    print("Conf min & max:", confs.min(), confs.max())
    confs = (confs - confs.min()) / (confs.max() - confs.min() + 1e-8)
    iio.mimwrite("temp_conf.mp4", tensor_to_video(confs[:, None].repeat(1, 3, 1, 1)), fps=16)


In [ ]:
import imageio.v2 as iio

render_images = render_pt3d_points(288, 512,
    points.to("cuda"), colors.to("cuda"), data["C2W"][I].to("cuda"), data["fxfycxcy"][I].to("cuda"))
iio.mimwrite("temp_render.mp4", tensor_to_video(render_images.cpu()), fps=30)

# Visualize DA3

In [ ]:
import imageio.v2 as iio
import numpy as np
from decord import VideoReader, cpu
import torchvision.transforms as tvT
from src.options import ROOT
from src.utils import *

x = np.load(f"{ROOT}/data/RealCam-Vid-DA3/RealEstate10K/test/dbFGT4L8aBw/651dda44a5c1e293.npz")
vr = VideoReader(str(f"{ROOT}/data/RealCam-Vid/RealEstate10K/test/dbFGT4L8aBw/651dda44a5c1e293.mp4"), ctx=cpu(0))
v = vr[:].asnumpy()

depths, confs, W2C, intrinsics = x["depth"], x["conf"], x["extrinsics"], x["intrinsics"]
print(depths.shape, confs.shape, W2C.shape, intrinsics.shape)

v = torch.from_numpy(v).permute(0, 3, 1, 2) / 255.
v = tvT.Resize((280, 504), tvT.InterpolationMode.BICUBIC)(v).clamp(0., 1.)
print(v.shape)


In [ ]:
depths = torch.from_numpy(depths).float()
confs = torch.from_numpy(confs).float()

W2C_ = torch.eye(4).unsqueeze(0).repeat(W2C.shape[0], 1, 1)
W2C_[:, :3, :4] = torch.from_numpy(W2C).float()
C2W = inverse_c2w(W2C_)
intrinsics[:, 0, 0] /= 504
intrinsics[:, 1, 1] /= 280
intrinsics[:, 0, 2] /= 504
intrinsics[:, 1, 2] /= 280
fxfycxcy = intrinsics_to_fxfycxcy(torch.from_numpy(intrinsics).float()[None, ...])[0]

In [ ]:
iio.mimwrite("temp.mp4", tensor_to_video(v), fps=30)
iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(1./depths)), fps=30)
iio.mimwrite("temp_conf.mp4", tensor_to_video(colorize_depth(confs)), fps=30)

xyzs = unproject_depth(depths[None, ...], C2W[None, ...], fxfycxcy[None, ...])[0]
xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
xyzs = rearrange(xyzs, "c f h w -> f c h w")
iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs), fps=30)

save_xyz_rgb_as_ply("temp.ply", xyzs, v, ratio=0.02)

# Save EMA Checkpoint

In [ ]:
import torch
from src.options import opt_dict, ROOT
from src.models import MyEMAModel
from src.models import Wan

opt = opt_dict["wan2.1_t2v_1.3b"]
model = Wan(opt)

tag, iter = "wan_t2v_cc_81x512", 15689

ema_path = f"{ROOT}/projects/VGGM/out/{tag}/checkpoints/{iter:06d}/ema_states.pth"
ema_states = MyEMAModel(model.parameters())
ema_states.load_state_dict(torch.load(ema_path, map_location="cpu", weights_only=True))
ema_states.copy_to(model.parameters())

torch.save(model.diffusion.state_dict(), f"{tag}_{iter:06d}.pth")